In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from sentence_transformers import CrossEncoder

import os

In [2]:
loader = DirectoryLoader('data', glob="*.txt", show_progress=True, loader_cls=TextLoader)

data = loader.load()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 415.24it/s]


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(data)

In [4]:
print("Number of Documents:", len(data))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 3

Number of Chunks: 181


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(collection_name="vector_database",
            embedding_function=embedding_model,
            persist_directory="./chroma_db_")
db.add_documents(documents=chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['0f8d0a44-019c-46a3-a46d-cc7605229ad2',
 '344563e3-0db8-4e52-a603-47e20554249d',
 '037ec148-5fc4-4cc5-8bcf-b80cba2a2f7b',
 '9fcf4558-01b4-4897-83dd-e9bb1bab6c09',
 '9f182aac-d412-4e80-bbb9-7f2fc361b229',
 '03569f87-9977-4250-94db-81ea8b56ad53',
 '824991f7-311e-4bab-a89f-d0d605064c14',
 '71577fd1-882c-41b2-8f06-3cf9c20b1bde',
 '8912969e-30cb-4206-83cb-801381f768d2',
 '27d4c592-1497-4609-980b-354b87d420c5',
 'f08269e6-4a63-415e-91f3-fb2e319849be',
 'd77ed3df-e2f1-4003-86df-42c10acdef3c',
 '7bf3e212-7596-4f8d-b172-62101b05df81',
 '3efd813f-44a3-4b5d-9dc0-d1491a9b6900',
 '6dd2c25a-018f-47af-b430-ec12825732a9',
 '2020e47a-9217-47b3-8bdd-48451b6da912',
 'e9926007-8073-4395-935e-4dd82df55085',
 '5eca0b29-5a7c-4f49-adc1-03b6953f2606',
 '59b4e319-abb4-4190-aaff-164b64d20af2',
 'f0e448ec-c008-4b9e-9bfb-f47794275e9b',
 '24c67760-3e3e-4adb-a113-0c76f159afc5',
 'e1aeb153-715d-458c-a70a-681d1ddbe610',
 'c5944686-15ff-429d-a879-d3e5b7f4366e',
 '7724b464-38c4-4963-a047-f76622b68d24',
 'bc103e9c-0482-

In [6]:
print(len(db.get()["ids"]))

362


In [7]:
dense_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [8]:
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 5

In [9]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

In [12]:
f = open('keys/groq_api_key.txt')
API_KEY = f.read()

In [13]:
groq_chat_model = ChatGroq(api_key=API_KEY,
                           model="openai/gpt-oss-20b",
                           temperature=1)

In [14]:
MULTI_QUERY_PROMPT = """You are an AI assistant. Your task is to generate {n} different versions
of the given user question to retrieve relevant documents from a knowledge base.
Generate alternative versions that capture different phrasings or perspectives of the same intent.
Return only the questions, one per line, with no numbering or extra text.

Original question: {question}"""

multi_query_prompt_template = ChatPromptTemplate.from_template(MULTI_QUERY_PROMPT)

In [15]:
parser = StrOutputParser()

multi_query_chain = multi_query_prompt_template | groq_chat_model | parser

def generate_queries(question: str, n: int = 3) -> list[str]:
    response = multi_query_chain.invoke({"question": question, "n": n})
    queries = [q.strip() for q in response.strip().split("\n") if q.strip()]
    return [question] + queries

In [16]:
def multi_query_hybrid_retrieve(question: str) -> list:
    queries = generate_queries(question)

    all_docs = []
    seen_contents = set()

    for query in queries:
        docs = hybrid_retriever.invoke(query)
        for doc in docs:
            if doc.page_content not in seen_contents:
                seen_contents.add(doc.page_content)
                all_docs.append(doc)

    return all_docs

In [17]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

def rerank_documents(question: str, docs: list, top_k: int = 5) -> list:
    pairs = [[question, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored_docs[:top_k]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

D:\Learning\AI\langchain_environment\.lc-exp-venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pkracham\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [18]:
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don't justify your answers.
Don't give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

In [19]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

generator_chain = prompt_template | groq_chat_model | parser

In [20]:
def hybrid_rag_pipeline(question: str) -> str:
    retrieved_docs = multi_query_hybrid_retrieve(question)
    reranked_docs = rerank_documents(question, retrieved_docs, top_k=5)
    context = format_docs(reranked_docs)
    response = generator_chain.invoke({"context": context, "question": question})
    return response

In [21]:
query = "What is the primary characteristic of Alzheimer's disease?"

hybrid_rag_pipeline(query)

"The primary characteristic of Alzheimer's disease is difficulty in remembering recent events, reflecting progressive memory impairment that originates from damage to the hippocampus."